# 01 · Preprocesamiento y construcción de series temporales
**Proyecto ML — Bizkaibus 2026**

Este notebook consolida los CSV de Open Data Bizkaia en dos datasets limpios:
- `train_viajeros.csv` → serie mensual de viajeros por línea (2020–2025)
- `train_expediciones.csv` → serie anual de expediciones por línea (2021–2025)
- `train_merged.csv` → dataset unificado para modelado


In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


## 1. Carga y consolidación — Viajeros (Bizkaibus)

In [2]:
# Carga de todos los CSV de viajeros
files_biz = sorted(glob.glob('../data/bizkaibus/*.csv'))
print(f"Archivos encontrados: {len(files_biz)}")

dfs = []
for f in files_biz:
    df = pd.read_csv(f, encoding='utf-8-sig')
    df.columns = [c.strip() for c in df.columns]
    dfs.append(df)

biz_raw = pd.concat(dfs, ignore_index=True)
print(f"Shape total: {biz_raw.shape}")
biz_raw.head(3)


Archivos encontrados: 18
Shape total: (6852, 44)


,_id,EKITALDIA/EJERCICIO,HILABETE/MES,ZENBAKIA/CODIGO,LERROA/LINEA,CREDITRANS/CREDITRANS,CREDITRANS F20/CREDITRANS F20,CREDITRANS F50/CREDITRANS F50,GIZATRANS/GIZATRANS,GIZATRANS 20/GIZATRANS 20,...,BORO/BORO,BORO F20/BORO F20,BORO F50/BORO F50,BAT/BAT,BAT F20/BAT F20,BAT F50/BAT F50,BAT BEREZI/BAT BEREZI,BAT BEREZI F20/BAT BEREZI F20,BAT BEREZI F50/BAT BEREZI F50,ITSU-MUPEN DOAKO GIDARILAGUN/ACOMPANANTE GRATIS INVIDENTE-PMRS
0,1,2020,7,A2610,GALDAKAO - UPV/EHU,864.0,4.0,1.0,51.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2020,7,A2611,UGAO MIRABALLES - BASAURI - ETXEBARRI - UPV/EHU,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2020,7,A3613,BILBAO - UGAO MIRABALLES - OROZKO,14240.0,274.0,73.0,6329.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Renombrado de columnas clave (snake_case)
biz = biz_raw.rename(columns={
    'EKITALDIA/EJERCICIO': 'año',
    'HILABETE/MES':        'mes',
    'ZENBAKIA/CODIGO':     'codigo_linea',
    'LERROA/LINEA':        'linea',
    'GUZTIRA/TOTAL':       'viajeros_total',
    'CREDITRANS/CREDITRANS':   'creditrans',
    'GIZATRANS/GIZATRANS':     'gizatrans',
    'GAZTE 70/GAZTE 70':       'gazte_70',
    'GORO/GORO':               'goro',
    'OHIKOA/OCASIONAL':        'ocasional',
    'FAMILIA UGARIA/FAMILIA NUMEROSA': 'familia_numerosa',
    '6 URTETIK BEHERAKOAK/MENORES DE 6 ANOS': 'menores_6',
    'ENPRESA/EMPRESA':         'empresa',
})

# Columnas de titulos jovenes (para H1 del EDA)
cols_jovenes = [c for c in biz.columns if 'GAZTE' in c or 'gazte' in c.lower()]

# Selección de columnas útiles
keep_cols = ['año','mes','codigo_linea','linea','viajeros_total',
             'creditrans','gizatrans','gazte_70','goro','ocasional',
             'familia_numerosa','menores_6','empresa']
keep_cols = [c for c in keep_cols if c in biz.columns]
biz = biz[keep_cols].copy()

print("Shape tras selección de columnas:", biz.shape)
print("Nulos por columna:")
print(biz.isnull().sum())


Shape tras selección de columnas: (6852, 13)
Nulos por columna:
año                   0
mes                   0
codigo_linea          0
linea                 0
viajeros_total        0
creditrans          194
gizatrans           278
gazte_70            570
goro                581
ocasional           149
familia_numerosa    625
menores_6           134
empresa              74
dtype: int64


In [4]:
# Limpieza: conversión numérica, eliminar filas con viajeros nulos
biz['viajeros_total'] = pd.to_numeric(biz['viajeros_total'], errors='coerce')
biz = biz.dropna(subset=['viajeros_total'])
biz = biz[biz['viajeros_total'] >= 0]

# Crear columna de fecha (primer día del mes)
biz['fecha'] = pd.to_datetime(biz['año'].astype(str) + '-' + biz['mes'].astype(str).str.zfill(2) + '-01')

# Flag COVID: año 2020 y 2021
biz['flag_covid'] = (biz['año'].isin([2020, 2021])).astype(int)

# Flag subsidio: desde septiembre 2022
biz['flag_subsidio'] = ((biz['año'] > 2022) | 
                         ((biz['año'] == 2022) & (biz['mes'] >= 9))).astype(int)

print("Rango de fechas:", biz['fecha'].min(), "→", biz['fecha'].max())
print("Líneas únicas:", biz['linea'].nunique())
print("Flag subsidio dist:")
print(biz.groupby('flag_subsidio')['fecha'].agg(['min','max','count']))


Rango de fechas: 2020-01-01 00:00:00 → 2025-12-01 00:00:00
Líneas únicas: 104
Flag subsidio dist:
                     min        max  count
flag_subsidio                             
0             2020-01-01 2022-08-01   3052
1             2022-09-01 2025-12-01   3800


In [5]:
# Verificar cobertura completa: todas las líneas deben tener 72 registros (6 años × 12 meses)
cobertura = biz.groupby('linea')['fecha'].count()
print("Distribución de registros por línea:")
print(cobertura.value_counts().sort_index())
print()
# Líneas con cobertura incompleta
incompletas = cobertura[cobertura < 72]
if len(incompletas) > 0:
    print(f"Líneas con cobertura incompleta ({len(incompletas)}):")
    print(incompletas.sort_values())
else:
    print("✓ Todas las líneas tienen cobertura completa.")


Distribución de registros por línea:
fecha
3      3
6      1
12     1
24     4
48     4
66     1
69     3
72    87
Name: count, dtype: int64

Líneas con cobertura incompleta (17):
linea
ERRIGOITI - GERNIKA LUMO - EREÃO                      3
GERNIKA LUMO - EA                                      3
MORGA - GERNIKA LUMO - MENDATA                         3
BILBAO - Sodupe - Arespalditza RESPALDIZA              6
Astrabudua-erandiogoikoa-SONDIKA                      12
LEIOA-Hospital Urduliz Ospitalea - GATIKA -MUNGIA     24
BILBAO-Corredor Txorierri-ko Korridorea-LARRABETZU    24
BILBAO - LAUKIZ - MUNGIA                              24
MUNGIA-DERIO-Gurutzeta/Cruces                         24
MUNGIA - DERIO - Cruces/Gurutzeta                     48
BILBAO-LAUKIZ                                         48
BILBAO-LOIU-Lauroeta-DERIO                            48
SOPELANA - MUNGIA - GATIKA                            48
BILBAO - Sodupe - Arespalditza Respaldiza             66
GERNIKA LUMO - E

## 2. Carga y consolidación — Expediciones

In [6]:
# Carga de todos los CSV de expediciones
files_exp = sorted(glob.glob('../data/expediciones/*.csv'))
print(f"Archivos encontrados: {len(files_exp)}")

dfs_exp = []
for f in files_exp:
    df = pd.read_csv(f, encoding='utf-8-sig')
    df.columns = [c.strip() for c in df.columns]
    dfs_exp.append(df)

exp_raw = pd.concat(dfs_exp, ignore_index=True)
print(f"Shape total: {exp_raw.shape}")
exp_raw.head(3)


Archivos encontrados: 5
Shape total: (8598, 14)


,_id,EKITALDIA/EJERCICIO,DATUAK ERAUZI DIREN EGUNA/FECHA EXTRACCION,LINEA_KODEA/CODIGO LINEA,LINEA/LINEA,DENBORALDI/TEMPORADA,EGUN MOTA/TIPO DIA,KOPURUA/NUMERO DIAS,ESPEDIZIOAK EGUNEAN/EXPEDICIONES AL DIA,ESPEDIZIOAK GUZTIRA/TOTAL EXPEDICIONES,JOANAK EGUNEAN/IDAS AL DIA,JOANAK GUZTIRA/TOTAL IDAS,ITZULIAK EGUNEAN/VUELTAS AL DIA,ITZULIAK GUZTIRA/TOTAL VUELTAS
0,333,2021,2021-09-15 08:37:19,A0652,LANESTOSA - BALMASEDA,Negua,Laborable,123,14,1722,7,861,7,861
1,328,2021,2021-09-15 08:37:19,A0651,BILBAO - BALMASEDA,Negua,Festivo,32,30,960,15,480,15,480
2,329,2021,2021-09-15 08:37:19,A0651,BILBAO - BALMASEDA,Negua,Laborable,123,60,7380,30,3690,30,3690


In [7]:
# Renombrado
exp = exp_raw.rename(columns={
    'EKITALDIA/EJERCICIO':               'año',
    'LINEA_KODEA/CODIGO LINEA':          'codigo_linea',
    'LINEA/LINEA':                       'linea',
    'DENBORALDI/TEMPORADA':              'temporada',
    'EGUN MOTA/TIPO DIA':                'tipo_dia',
    'KOPURUA/NUMERO DIAS':               'num_dias',
    'ESPEDIZIOAK EGUNEAN/EXPEDICIONES AL DIA': 'exp_por_dia',
    'ESPEDIZIOAK GUZTIRA/TOTAL EXPEDICIONES':  'expediciones_total',
})

# Agregación anual por línea (suma de todas las temporadas y tipos de día)
exp_anual = exp.groupby(['año','codigo_linea','linea'], as_index=False).agg(
    expediciones_anuales=('expediciones_total', 'sum'),
    dias_servicio=('num_dias', 'sum'),
)

print("Expediciones anuales shape:", exp_anual.shape)
print("Años disponibles:", sorted(exp_anual['año'].unique()))
print("Líneas únicas:", exp_anual['linea'].nunique())
exp_anual.head()


Expediciones anuales shape: (526, 5)
Años disponibles: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Líneas únicas: 109


,año,codigo_linea,linea,expediciones_anuales,dias_servicio
0,2021,A0651,BILBAO - BALMASEDA,46167,909
1,2021,A0652,LANESTOSA - BALMASEDA,9946,909
2,2021,A0653,TRUCIOS TURTZIOZ - ARTZENTALES,8736,624
3,2021,A0654,BALMASEDA - Gurutzeta/Cruces - UPV/EHU,16222,624
4,2021,A2151,Areeta/Las Arenas - LARRABETZU,26274,909


## 3. Construcción del dataset unificado

In [8]:
# Agregación mensual de viajeros (ya está a nivel mes-línea, solo ordenamos)
viajeros_mensual = biz.sort_values(['linea','fecha']).reset_index(drop=True)

# Guardar dataset de viajeros
os.makedirs('../data', exist_ok=True)
viajeros_mensual.to_csv('../data/train_viajeros.csv', index=False)
print(f"✓ train_viajeros.csv guardado → {viajeros_mensual.shape}")

# Guardar dataset de expediciones
exp_anual.to_csv('../data/train_expediciones.csv', index=False)
print(f"✓ train_expediciones.csv guardado → {exp_anual.shape}")


✓ train_viajeros.csv guardado → (6852, 16)
✓ train_expediciones.csv guardado → (526, 5)


In [9]:
# Merge: unir viajeros con expediciones anuales
# Las expediciones son anuales, las propagamos a cada mes del año correspondiente
merged = viajeros_mensual.merge(
    exp_anual[['año','linea','expediciones_anuales','dias_servicio']],
    on=['año','linea'],
    how='left'
)

# Expediciones mensuales estimadas (expediciones_anuales / 12)
merged['expediciones_mes_est'] = merged['expediciones_anuales'] / 12

# Ratio de saturación: viajeros / expediciones mensuales estimadas
merged['ratio_saturacion'] = merged['viajeros_total'] / merged['expediciones_mes_est'].replace(0, np.nan)

print("Dataset merged shape:", merged.shape)
print(f"Cobertura expediciones: {merged['expediciones_anuales'].notna().sum()} / {len(merged)} filas ({merged['expediciones_anuales'].notna().mean()*100:.1f}%)")
print()
print("Nulos en columnas clave:")
print(merged[['viajeros_total','expediciones_anuales','ratio_saturacion']].isnull().sum())
merged.head()


Dataset merged shape: (6852, 20)
Cobertura expediciones: 4431 / 6852 filas (64.7%)

Nulos en columnas clave:
viajeros_total             0
expediciones_anuales    2421
ratio_saturacion        2421
dtype: int64


,año,mes,codigo_linea,linea,viajeros_total,creditrans,gizatrans,gazte_70,goro,ocasional,familia_numerosa,menores_6,empresa,fecha,flag_covid,flag_subsidio,expediciones_anuales,dias_servicio,expediciones_mes_est,ratio_saturacion
0,2020,1,A3928,ARTEA - ZEBERIO - UGAO MIRABALLES,1999.0,1020.0,725.0,4.0,4.0,167.0,0.0,20.0,13.0,2020-01-01,1,0,NaN,NaN,NaN,NaN
1,2020,2,A3928,ARTEA - ZEBERIO - UGAO MIRABALLES,2118.0,1092.0,735.0,11.0,2.0,211.0,0.0,13.0,7.0,2020-02-01,1,0,NaN,NaN,NaN,NaN
2,2020,3,A3928,ARTEA - ZEBERIO - UGAO MIRABALLES,1019.0,542.0,365.0,1.0,3.0,75.0,0.0,9.0,7.0,2020-03-01,1,0,NaN,NaN,NaN,NaN
3,2020,4,A3928,ARTEA - ZEBERIO - UGAO MIRABALLES,242.0,147.0,93.0,1.0,1.0,0.0,0.0,0.0,0.0,2020-04-01,1,0,NaN,NaN,NaN,NaN
4,2020,5,A3928,ARTEA - ZEBERIO - UGAO MIRABALLES,663.0,383.0,191.0,34.0,48.0,0.0,0.0,0.0,0.0,2020-05-01,1,0,NaN,NaN,NaN,NaN


## 4. Feature Engineering base

In [10]:
# Features temporales
merged['trimestre']     = merged['fecha'].dt.quarter
merged['mes_num']       = merged['fecha'].dt.month
merged['año_num']       = merged['fecha'].dt.year
merged['mes_sin']       = np.sin(2 * np.pi * merged['mes_num'] / 12)
merged['mes_cos']       = np.cos(2 * np.pi * merged['mes_num'] / 12)

# Features de lag (por línea, ordenado por fecha)
merged = merged.sort_values(['linea','fecha']).reset_index(drop=True)

for linea, grupo in merged.groupby('linea'):
    idx = grupo.index
    merged.loc[idx, 'lag_1']  = grupo['viajeros_total'].shift(1).values
    merged.loc[idx, 'lag_3']  = grupo['viajeros_total'].shift(3).values
    merged.loc[idx, 'lag_12'] = grupo['viajeros_total'].shift(12).values
    merged.loc[idx, 'rolling_mean_3']  = grupo['viajeros_total'].shift(1).rolling(3).mean().values
    merged.loc[idx, 'rolling_mean_6']  = grupo['viajeros_total'].shift(1).rolling(6).mean().values
    merged.loc[idx, 'rolling_mean_12'] = grupo['viajeros_total'].shift(1).rolling(12).mean().values

print("Features de lag añadidas.")
print("Nulos introducidos por lag (esperado):")
print(merged[['lag_1','lag_3','lag_12','rolling_mean_3','rolling_mean_6','rolling_mean_12']].isnull().sum())


Features de lag añadidas.
Nulos introducidos por lag (esperado):
lag_1               104
lag_3               312
lag_12             1215
rolling_mean_3      312
rolling_mean_6      615
rolling_mean_12    1215
dtype: int64


In [11]:
# Guardar dataset final
merged.to_csv('../data/train_merged.csv', index=False)
print(f"✓ train_merged.csv guardado → {merged.shape}")
print()
print("=== Resumen del dataset final ===")
print(f"  Período:         {merged['fecha'].min().date()} → {merged['fecha'].max().date()}")
print(f"  Líneas:          {merged['linea'].nunique()}")
print(f"  Filas totales:   {len(merged):,}")
print(f"  Columnas:        {len(merged.columns)}")
print(f"  Pre-subsidio:    {(merged['flag_subsidio']==0).sum():,} registros")
print(f"  Post-subsidio:   {(merged['flag_subsidio']==1).sum():,} registros")
print()
merged.describe()


✓ train_merged.csv guardado → (6852, 31)

=== Resumen del dataset final ===
  Período:         2020-01-01 → 2025-12-01
  Líneas:          104
  Filas totales:   6,852
  Columnas:        31
  Pre-subsidio:    3,052 registros
  Post-subsidio:   3,800 registros



,año,mes,viajeros_total,creditrans,gizatrans,gazte_70,goro,ocasional,familia_numerosa,menores_6,...,mes_num,año_num,mes_sin,mes_cos,lag_1,lag_3,lag_12,rolling_mean_3,rolling_mean_6,rolling_mean_12
count,6852.000000,6852.000000,6852.000000,6658.000000,6574.000000,6282.000000,6271.000000,6703.000000,6227.000000,6718.000000,...,6852.000000,6852.000000,6.852000e+03,6.852000e+03,6748.000000,6540.000000,5637.000000,6540.000000,6237.000000,5637.000000
mean,2022.495622,6.500000,21999.843831,14326.599130,3446.220679,954.399975,856.876988,1117.185134,2.565762,326.556862,...,6.500000,2022.495622,-1.367525e-17,-4.562738e-17,21880.448123,21590.147849,20582.863391,21741.703710,21782.697146,21941.385846
min,2020.000000,1.000000,0.000000,0.000000,0.000000,-2908.000000,-2164.000000,0.000000,0.000000,0.000000,...,1.000000,2020.000000,-1.000000e+00,-1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2021.000000,3.750000,3615.750000,2711.500000,308.000000,121.000000,129.000000,51.500000,0.000000,9.000000,...,3.750000,2021.000000,-5.915064e-01,-5.915064e-01,3587.500000,3536.000000,3332.000000,4205.583333,4523.000000,4640.916667
50%,2022.000000,6.500000,13579.500000,9357.000000,1768.500000,570.000000,471.000000,353.000000,0.000000,108.000000,...,6.500000,2022.000000,-6.123234e-17,-6.123234e-17,13543.500000,13376.500000,12745.000000,13650.166667,13876.166667,14108.750000
75%,2024.000000,9.250000,33415.250000,22395.500000,5077.250000,1359.750000,1150.500000,1160.000000,2.000000,398.750000,...,9.250000,2024.000000,5.915064e-01,5.915064e-01,33164.000000,32800.250000,31708.000000,32209.500000,32457.000000,32832.250000
max,2025.000000,12.000000,159804.000000,85327.000000,27947.000000,10900.000000,10064.000000,71335.000000,94.000000,5135.000000,...,12.000000,2025.000000,1.000000e+00,1.000000e+00,159804.000000,149772.000000,145608.000000,154137.333333,147168.833333,143033.916667
std,1.709653,3.452304,24773.520542,15021.696743,4246.304487,1146.232146,1123.578975,4129.276078,7.160563,590.594265,...,3.452304,1.709653,7.071584e-01,7.071584e-01,24614.972384,24263.832477,22948.143591,23952.822479,23569.229716,23246.909580


## 5. Split train / test para evaluación de modelos

- **Train**: enero 2020 → diciembre 2024
- **Test (hold-out)**: enero 2025 → diciembre 2025
- **Contrafactual**: enero 2020 → agosto 2022 (pre-subsidio)


In [12]:
# Split temporal
train = merged[merged['año'] <= 2024].copy()
test  = merged[merged['año'] == 2025].copy()
pre_subsidio = merged[merged['flag_subsidio'] == 0].copy()

train.to_csv('../data/train.csv', index=False)
test.to_csv('../data/test.csv',   index=False)
pre_subsidio.to_csv('../data/pre_subsidio.csv', index=False)

print(f"✓ train.csv          → {train.shape[0]:,} filas  ({train['fecha'].min().date()} – {train['fecha'].max().date()})")
print(f"✓ test.csv           → {test.shape[0]:,} filas   ({test['fecha'].min().date()} – {test['fecha'].max().date()})")
print(f"✓ pre_subsidio.csv   → {pre_subsidio.shape[0]:,} filas  ({pre_subsidio['fecha'].min().date()} – {pre_subsidio['fecha'].max().date()})")


✓ train.csv          → 5,712 filas  (2020-01-01 – 2024-12-01)
✓ test.csv           → 1,140 filas   (2025-01-01 – 2025-12-01)
✓ pre_subsidio.csv   → 3,052 filas  (2020-01-01 – 2022-08-01)
